# D8.1 — Stage 2d Aggregate Result Analysis

Post-fire analytical adjudication of the Stage 2d 200-call batch (`batch_5cf76668-47d1-48d7-bd90-db06d31982ed`) against the pre-registered Scope Lock v2 claims in `docs/d7_stage2d/stage2d_expectations.md`.

This notebook is the first executable output of D8 (Stage 2d Result Adjudication). Downstream notebooks (D8.2, D8.3) consume its verdict.

## Scope, non-scope, hard rules

**Scope.** Gate adjudication for the five pre-registered claims (§6.2.1, §6.2.2, §6.3(a), §6.3(b), §6.4) and readout-only recording of §6.6 observation axes (alignment, SVR–alignment decouple, theme × label contingency, plausibility distribution).

**Non-scope.** §E3 Tier 1/2 bucket adjudication (D8.2). Axis-reversal narrative framing of §6.2.2 (D8.2 interpretation layer). Test-retest and drift magnitude analysis (D8.3). Any re-derivation of the fresh-7 subset at runtime.

**Hard rules (seven, binding on every cell below):**

1. Scoring universe is the 197 `critic_status == "ok"` records. Positions 42 and 87 (`d7b_error`) and position 116 (`skipped_source_invalid` synthetic pos 116) are quarantined, never imputed.
2. No imputation. A missing field, NaN, or absent key is a keyError — cells must raise, not substitute.
3. Pre-registered thresholds are honoured literally. No post-hoc reframing, threshold inversion, or direction swap.
4. §6.6 is readout-only — no PASS/FAIL adjudication on alignment, decouple, theme × label, plausibility, or neutral-stratum median.
5. The fresh-7 subset is the hard-coded literal `{3, 43, 68, 128, 173, 188, 198}`. Never re-derived from `deep_dive_candidates.json` or factor-set parsing.
6. All input artifacts are SHA256-verified against frozen anchors before any analytical cell executes.
7. Denominator semantics are explicit in every cell. 199 for §6.3 tail counts (UB full). 66 for §6.2.1 agreement. 5 for §6.2.2 divergence. 7 for §6.4 fresh-7. 197 for §6.6 observation readouts.

**Label-join invariants (two, inherited from Stage 2c):**

a. `universe_b_label` is the authoritative axis label for gate adjudication. `universe_a_label` is metadata only.
b. Per-call records are keyed by `candidate_position` except the synthetic pos 116, which uses `position` (Lock 1.5).

In [ ]:
# Cell 02 — imports, repo-root resolver, path pins, expected SHA anchors, helpers.

import hashlib
import json
from pathlib import Path

# Repo-root resolver: notebook lives at docs/test_notebooks/, walk up until we find
# a known repo-root sentinel (CLAUDE.md). Supports nbclient + direct Jupyter runs.
def _resolve_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise RuntimeError(f"repo root not found from cwd={here}")

REPO_ROOT = _resolve_repo_root()

BATCH_DIR = REPO_ROOT / "raw_payloads" / "batch_5cf76668-47d1-48d7-bd90-db06d31982ed"
CRITIC_DIR = BATCH_DIR / "critic"

AGGREGATE_PATH      = CRITIC_DIR / "stage2d_aggregate_record.json"
STARTUP_AUDIT_PATH  = CRITIC_DIR / "stage2d_startup_audit.json"
EXPECTATIONS_PATH   = REPO_ROOT / "docs" / "d7_stage2d" / "stage2d_expectations.md"
REPLAY_CANDIDATES_PATH = REPO_ROOT / "docs" / "d7_stage2d" / "replay_candidates.json"
SCOPE_LOCK_PATH     = REPO_ROOT / "docs" / "d7_stage2d" / "D7_STAGE2D_SCOPE_LOCK_v2.md"

EXPECTED_SHA256 = {
    "stage2d_aggregate_record.json": "09eeda3278c96ccf7b945c5edc9dde9bcfa51ca35138a63d36258514be5c323f",
    "stage2d_startup_audit.json":    "4f46de0902532c3d22d22d06c9f5604e82eff332b47f3aabf1603ef81e8a9ea5",
    "stage2d_expectations.md":       "98b87a702cc80df2d993d51857d4142f93f2ab8be66438bd2937c5dd374010a5",
    "replay_candidates.json":        "05706642cbeb5014d5072c172b343b01ee56d0eb8ee45afb2f3ce6e56665907e",
    "D7_STAGE2D_SCOPE_LOCK_v2.md":   "b4cad5873707c6eba272d313e0214011cb5ca91b142126013946f91a72496382",
}

# Hard denominators — asserted throughout the notebook.
STAGE2D_SOURCE_N = 200            # all per-call records
STAGE2D_LIVE_N   = 199            # live D7b attempted / non-skipped
STAGE2D_OK_N     = 197            # successfully scored (critic_status == "ok")
STAGE2D_D7B_ERR_N = 2             # pos 42, 87
STAGE2D_SKIPPED_N = 1             # pos 116 synthetic

ERROR_POSITIONS = (42, 87)
SKIPPED_POSITIONS = (116,)
QUARANTINE_POSITIONS = frozenset(ERROR_POSITIONS + SKIPPED_POSITIONS)

# Fresh-7 RSI-absent vol_regime — hard-coded literal per §6.4, hard rule 5.
FRESH_7_RSI_ABSENT = frozenset({3, 43, 68, 128, 173, 188, 198})
assert len(FRESH_7_RSI_ABSENT) == 7, "fresh-7 literal must be 7 positions"

# Stage 2b overlap positions — per expectations.md §E3 Tier 1 anchor.
STAGE2B_OVERLAP_POSITIONS = frozenset({17, 73, 74, 97, 138})

# Helpers.
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

checks: list[tuple[str, bool, str]] = []

def check(label: str, condition: bool, observed: str) -> None:
    """Append a PASS/FAIL row with observed value."""
    checks.append((label, bool(condition), observed))
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {label} -- observed={observed}")

def check_equal(label: str, observed, expected) -> None:
    """Append a PASS/FAIL row comparing observed to expected."""
    ok = observed == expected
    checks.append((label, ok, f"observed={observed!r} expected={expected!r}"))
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {label} -- observed={observed!r}; expected={expected!r}")

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"AGGREGATE_PATH exists: {AGGREGATE_PATH.exists()}")
print(f"5 input paths resolved; {len(EXPECTED_SHA256)} SHA anchors pinned.")


In [ ]:
# Cell 03 — SHA256 verification for aggregate + 4 secondary inputs.

for path in (
    AGGREGATE_PATH,
    STARTUP_AUDIT_PATH,
    EXPECTATIONS_PATH,
    REPLAY_CANDIDATES_PATH,
    SCOPE_LOCK_PATH,
):
    observed = sha256_of(path)
    expected = EXPECTED_SHA256[path.name]
    check_equal(f"sha256({path.name})", observed, expected)

# Hard failure if any SHA mismatches — adjudication is void without frozen inputs.
assert all(ok for _, ok, _ in checks), "SHA verification failed; adjudication aborted"


In [ ]:
# Cell 04 — load aggregate, validate 51-key shape, Lock 11 tail, status tally.

aggregate = load_json(AGGREGATE_PATH)

top_keys = list(aggregate.keys())
check_equal("aggregate top-level key count", len(top_keys), 51)
check_equal("aggregate Lock 11 tail key", top_keys[-1], "write_completed_at")

# Denominator surface on the aggregate itself.
check_equal("aggregate.stage2d_source_n",      aggregate["stage2d_source_n"],     STAGE2D_SOURCE_N)
check_equal("aggregate.stage2d_live_d7b_call_n", aggregate["stage2d_live_d7b_call_n"], STAGE2D_LIVE_N)
check_equal("aggregate.stage2d_skipped_positions", list(aggregate["stage2d_skipped_positions"]), list(SKIPPED_POSITIONS))
check_equal("aggregate.stage2b_overlap_positions (sorted)",
            sorted(aggregate["stage2b_overlap_positions"]),
            sorted(STAGE2B_OVERLAP_POSITIONS))

per_call_records = aggregate["per_call_records"]
check_equal("per_call_records length", len(per_call_records), STAGE2D_SOURCE_N)

critic_status_counts = aggregate["critic_status_counts"]
check_equal(
    "critic_status_counts partition",
    critic_status_counts,
    {"ok": STAGE2D_OK_N, "d7b_error": STAGE2D_D7B_ERR_N, "skipped_source_invalid": STAGE2D_SKIPPED_N},
)

d7b_scores_by_call = aggregate["d7b_scores_by_call"]
check_equal("d7b_scores_by_call entry count", len(d7b_scores_by_call), STAGE2D_SOURCE_N)

# Pos 42 / 87 / 116 must map to None in the scores dict (no SVR produced).
for pos in QUARANTINE_POSITIONS:
    check_equal(f"d7b_scores_by_call[str({pos})] is None", d7b_scores_by_call[str(pos)], None)


In [ ]:
# Cell 05 — build ok_frame (exactly 197 rows, critic_status == "ok").
# Root-level fields from per_call_records; score axes from d7b_scores_by_call[str(pos)].
# Schema note: d7b_scores_by_call preserves 200 positional slots, with
# quarantined records represented as None; the analytical scoring universe
# is constructed from per_call_records where critic_status == "ok".
# Hard rule 4: no try/except around dict lookups — KeyError is the correct failure mode.

ROOT_FIELDS = (
    "candidate_position",
    "candidate_theme",
    "pre_registered_label",
    "is_stage2b_overlap",
    "is_deep_dive_candidate",
    "test_retest_tier",
    "stratum_id",
    "prior_factor_sets_count",
    "theme_hint_factor_count",
    "firing_order",
    "call_index",
    "retry_count",
    "actual_cost_usd",
)

ok_frame: list[dict] = []
for rec in per_call_records:
    if rec["critic_status"] != "ok":
        continue
    position = rec["candidate_position"]
    scores = d7b_scores_by_call[str(position)]
    # scores is a dict with 3 axes; iteration order matters only for display.
    row = {field: rec[field] for field in ROOT_FIELDS}
    row["svr"]          = scores["structural_variant_risk"]
    row["alignment"]    = scores["semantic_theme_alignment"]
    row["plausibility"] = scores["semantic_plausibility"]
    ok_frame.append(row)

check_equal("ok_frame length", len(ok_frame), STAGE2D_OK_N)

ok_positions = {row["candidate_position"] for row in ok_frame}
check_equal("ok_frame unique positions count", len(ok_positions), STAGE2D_OK_N)

# Quarantine invariant: no error / skipped position is in ok_frame.
for pos in QUARANTINE_POSITIONS:
    check(f"pos {pos} absent from ok_frame", pos not in ok_positions, f"in_ok_frame={pos in ok_positions}")

# Label partition on ok_frame must match §6.2 denominators.
from collections import Counter
label_counts = Counter(row["pre_registered_label"] for row in ok_frame)
check_equal(
    "ok_frame pre_registered_label partition",
    dict(label_counts),
    {"agreement_expected": 64, "divergence_expected": 5, "neutral": 128},
)
# Note: agreement is 64 here (not 66) because pos 42 and 87 are agreement_expected
# candidates that quarantined as d7b_error. §6.2.1 gate denominator is 66 (universe),
# adjudicated in D8.1.4; the ok_frame itself is the 197-row scoring universe.


## §1 — Universe integrity

**Claim.** The Stage 2d batch returned 200 per-call records partitioning exactly as 197 `ok` + 2 `d7b_error` (pos 42, 87) + 1 `skipped_source_invalid` (pos 116 synthetic, Lock 1.5 17-key shape).

**Adjudication.** Assert the count partition literal and print the quarantine audit record for pos 42/87/116 (status code, error signature, raw payload path presence).

In [ ]:
# Cell 07 — universe integrity assertions + quarantine audit for pos 42 / 87 / 116.

# Restate the 200 -> 199 / 1 -> 197 / 2 partition.
check_equal("200 source = 199 live + 1 skipped",
            STAGE2D_LIVE_N + STAGE2D_SKIPPED_N, STAGE2D_SOURCE_N)
check_equal("199 live = 197 ok + 2 d7b_error",
            STAGE2D_OK_N + STAGE2D_D7B_ERR_N, STAGE2D_LIVE_N)
check_equal("200 source = 197 ok + 2 d7b_error + 1 skipped",
            STAGE2D_OK_N + STAGE2D_D7B_ERR_N + STAGE2D_SKIPPED_N, STAGE2D_SOURCE_N)

def _find_record(pos: int) -> dict:
    for rec in per_call_records:
        if rec.get("candidate_position") == pos or rec.get("position") == pos:
            return rec
    raise KeyError(f"record for position {pos} not found")

print()
print("=== Quarantine audit ===")
for pos in ERROR_POSITIONS:
    rec = _find_record(pos)
    check_equal(f"pos {pos} critic_status", rec["critic_status"], "d7b_error")
    raw_paths = rec.get("raw_payload_paths")
    assert isinstance(raw_paths, dict) and raw_paths, f"pos {pos} missing raw_payload_paths"
    print(
        f"  pos={pos}  status={rec['critic_status']}  "
        f"category={rec.get('d7b_error_category')!r}  "
        f"signature={rec.get('critic_error_signature')!r}  "
        f"retry_count={rec.get('retry_count')}  "
        f"raw_payload_paths_keys={sorted(raw_paths.keys())}"
    )

for pos in SKIPPED_POSITIONS:
    rec = _find_record(pos)
    check_equal(f"pos {pos} critic_status", rec["critic_status"], "skipped_source_invalid")
    check_equal(f"pos {pos} d7b_call_attempted", rec["d7b_call_attempted"], False)
    check_equal(f"pos {pos} d7b_error_category", rec["d7b_error_category"], "source_invalid")
    print(
        f"  pos={pos}  status={rec['critic_status']}  "
        f"skip_reason={rec.get('skip_reason')!r}"
    )

# Final integrity gate for Sections 0-1.
pass_count = sum(1 for _, ok, _ in checks if ok)
fail_count = sum(1 for _, ok, _ in checks if not ok)
print()
print(f"Sections 0-1 integrity checks: {pass_count} PASS, {fail_count} FAIL")
assert fail_count == 0, f"{fail_count} integrity checks failed"


## §2 — §6.2 Axis gates

**§6.2.1 Agreement axis (TBD-A1).** *Of the 66 UB agreement-labelled calls, ≥ 52 have SVR ≥ 0.5.* Operational pass = `count(svr >= 0.5) over {ub == "agreement_expected"} >= 52`.

**§6.2.2 Divergence axis (TBD-A2).** *Of the 5 UB divergence-labelled calls, ≥ 4 contradict the `divergence_expected` label with SVR ≥ 0.5.* Operational pass = `count(svr >= 0.5) over {ub == "divergence_expected"} >= 4`.

Adjudication is literal: observed count vs pre-registered threshold. PASS/FAIL only. Any interpretive framing (e.g. axis reversal) is D8.2 scope, not D8.1.

In [ ]:
# Cell 09 — §6.2.1 agreement axis gate.
# Pre-registered (expectations.md §6.2.1, L67/L73):
#   count(call.svr >= 0.5) over {universe_b_label == "agreement_expected"} >= 52
# Denominator: 66 (UB agreement cohort).
# §6.1 Lock: "All aggregate denominators ... refer to the 199 replay-eligible
#   positions"; pos 116 is the only exclusion. d7b_error records (pos 42, 87)
#   remain in the 66-cohort denominator but produced no SVR, so they
#   cannot satisfy svr >= 0.5 and contribute 0 to the numerator.

AGREEMENT_UB_DENOM_PREREG = 66
AGREEMENT_THRESHOLD = 52

agreement_rows = [row for row in ok_frame if row["pre_registered_label"] == "agreement_expected"]
check_equal("agreement scored cohort (ok_frame)", len(agreement_rows), 64)

# Quarantined UB agreement calls: pos 42, pos 87 (both d7b_error, both agreement_expected).
agreement_ub_errored = [
    rec["candidate_position"] for rec in per_call_records
    if rec.get("pre_registered_label") == "agreement_expected"
    and rec.get("critic_status") == "d7b_error"
]
check_equal("agreement UB errored positions", sorted(agreement_ub_errored), [42, 87])

# Literal adjudication: numerator over the 66-cohort denominator.
agreement_numerator = sum(1 for row in agreement_rows if row["svr"] >= 0.5)
agreement_verdict = "PASS" if agreement_numerator >= AGREEMENT_THRESHOLD else "FAIL"

print()
print("=== §6.2.1 agreement axis gate (literal pre-registration) ===")
print(f"  pre-registered denominator : {AGREEMENT_UB_DENOM_PREREG}  (UB agreement_expected cohort)")
print(f"  pre-registered threshold   : count(SVR >= 0.5) >= {AGREEMENT_THRESHOLD}")
print(f"  scored (ok) subset         : {len(agreement_rows)}")
print(f"  quarantined (d7b_error)    : {sorted(agreement_ub_errored)}  (contribute 0 to numerator)")
print(f"  observed numerator         : {agreement_numerator}")
print(f"  observed ratio             : {agreement_numerator}/{AGREEMENT_UB_DENOM_PREREG}")
print(f"  verdict                    : {agreement_verdict}")

# Transparency readout over the scored-only denominator (not a gate).
agreement_scored_only_rate = agreement_numerator / len(agreement_rows) if agreement_rows else 0.0
print(f"  transparency readout (scored-only n=64): {agreement_numerator}/{len(agreement_rows)}"
      f" = {agreement_scored_only_rate:.4f}  (not the pre-registered gate; reported for post-hoc clarity)")

check_equal("§6.2.1 agreement verdict derived", agreement_verdict, "PASS" if agreement_numerator >= 52 else "FAIL")


In [ ]:
# Cell 10 — §6.2.2 divergence axis gate.
# Pre-registered (expectations.md §6.2.2, L77/L85):
#   count(call.svr >= 0.5) over {universe_b_label == "divergence_expected"} >= 4
# Denominator: 5 (no quarantined divergence records).
# Literal gate adjudication only. Any interpretive framing of the result
# (axis-reversal, sample-size, etc.) is D8.2 scope, not D8.1.

DIVERGENCE_UB_DENOM_PREREG = 5
DIVERGENCE_THRESHOLD = 4

divergence_rows = [row for row in ok_frame if row["pre_registered_label"] == "divergence_expected"]
check_equal("divergence scored cohort (ok_frame)", len(divergence_rows), 5)

divergence_ub_errored = [
    rec["candidate_position"] for rec in per_call_records
    if rec.get("pre_registered_label") == "divergence_expected"
    and rec.get("critic_status") == "d7b_error"
]
check_equal("divergence UB errored positions", divergence_ub_errored, [])

divergence_numerator = sum(1 for row in divergence_rows if row["svr"] >= 0.5)
divergence_verdict = "PASS" if divergence_numerator >= DIVERGENCE_THRESHOLD else "FAIL"

print()
print("=== §6.2.2 divergence axis gate (literal pre-registration) ===")
print(f"  pre-registered denominator : {DIVERGENCE_UB_DENOM_PREREG}  (UB divergence_expected cohort)")
print(f"  pre-registered threshold   : count(SVR >= 0.5) >= {DIVERGENCE_THRESHOLD}")
print(f"  scored (ok) subset         : {len(divergence_rows)}")
print(f"  observed numerator         : {divergence_numerator}")
print(f"  observed ratio             : {divergence_numerator}/{DIVERGENCE_UB_DENOM_PREREG}")
print(f"  verdict                    : {divergence_verdict}")
print(f"  per-call SVRs              : {[(row['candidate_position'], row['svr']) for row in sorted(divergence_rows, key=lambda r: r['candidate_position'])]}")

# Literal framing only; interpretation deferred.
print(f"  note                       : literal pre-registration adjudication only; interpretation of result (axis-reversal, "
      f"universe-expansion effects, etc.) is D8.2 scope, not D8.1.")

check_equal("§6.2.2 divergence verdict derived", divergence_verdict, "PASS" if divergence_numerator >= 4 else "FAIL")


## §3 — §6.3 SVR distribution-shape gates

Two independent counts over the 199-call UB set (ok_frame plus anything with a numeric SVR; by construction = 197 ok records — the §6.3 denominator is **199** per expectations.md because pos 42 and pos 87 are UB members even though they lack an SVR score). The expectations doc states `over all 199 UB calls` — cell 12 honours that denominator exactly; d7b_error records contribute 0 to both tail counts (they have no SVR to compare).

**§6.3(a) upper tail.** ≥ 90 of 199 have SVR ≥ 0.80.

**§6.3(b) lower tail.** ≥ 30 of 199 have SVR ≤ 0.30.

Both must pass for §6.3 as a whole to pass.

In [ ]:
# Cell 12 — §6.3(a) upper-tail + §6.3(b) lower-tail gates.
# Pre-registered (expectations.md §6.3, L99-L111):
#   (a) count(svr >= 0.80) over 199-call UB set >= 90
#   (b) count(svr <= 0.30) over 199-call UB set >= 30
#   joint: both (a) and (b) must pass for §6.3 as a whole to pass.
# §6.1 Lock: denominator is the 199 D7b-call set; pos 116 excluded.
# d7b_error records (pos 42, 87) remain in denominator but have no SVR and
# thus contribute 0 to both tail counts.

UB_DENOM_PREREG = 199
UPPER_THRESHOLD = 90
LOWER_THRESHOLD = 30

# Construct the 199-call UB cohort: all per_call_records except pos 116.
ub_cohort = [
    rec for rec in per_call_records
    if not (rec.get("critic_status") == "skipped_source_invalid"
            and rec.get("position") == 116)
]
check_equal("UB 199-call cohort length", len(ub_cohort), UB_DENOM_PREREG)

# Count how many in the cohort are errored (contribute 0 to numerators).
ub_errored = [rec for rec in ub_cohort if rec.get("critic_status") == "d7b_error"]
check_equal("UB cohort d7b_error count", len(ub_errored), 2)

# ok_frame positions partition the scored subset (197); SVR lookup via d7b_scores_by_call.
upper_tail_count = sum(1 for row in ok_frame if row["svr"] >= 0.80)
lower_tail_count = sum(1 for row in ok_frame if row["svr"] <= 0.30)

upper_verdict = "PASS" if upper_tail_count >= UPPER_THRESHOLD else "FAIL"
lower_verdict = "PASS" if lower_tail_count >= LOWER_THRESHOLD else "FAIL"
joint_verdict = "PASS" if (upper_verdict == "PASS" and lower_verdict == "PASS") else "FAIL"

print()
print("=== §6.3 SVR distribution-shape gates (literal pre-registration) ===")
print(f"  pre-registered denominator : {UB_DENOM_PREREG}  (199-call UB cohort, pos 116 excluded)")
print(f"  quarantined in denominator : {sorted(r['candidate_position'] for r in ub_errored)}  (no SVR, contribute 0)")
print(f"  scored (ok) subset         : {len(ok_frame)}")
print()
print(f"  §6.3(a) upper tail: count(SVR >= 0.80) >= {UPPER_THRESHOLD}")
print(f"    observed                 : {upper_tail_count}/{UB_DENOM_PREREG}  → {upper_verdict}")
print(f"  §6.3(b) lower tail: count(SVR <= 0.30) >= {LOWER_THRESHOLD}")
print(f"    observed                 : {lower_tail_count}/{UB_DENOM_PREREG}  → {lower_verdict}")
print(f"  §6.3 joint verdict         : {joint_verdict}  (both (a) and (b) must pass)")

check_equal("§6.3(a) upper-tail verdict derived", upper_verdict, "PASS" if upper_tail_count >= 90 else "FAIL")
check_equal("§6.3(b) lower-tail verdict derived", lower_verdict, "PASS" if lower_tail_count >= 30 else "FAIL")


In [ ]:
# Cell 13 — SVR distribution readout over 197 scored records.
# Observation only; not a pre-registered hypothesis test.
# §6.3 gate adjudication is cell 12; this cell describes distribution shape.

svr_values = sorted(row["svr"] for row in ok_frame)
n = len(svr_values)
check_equal("SVR readout denominator", n, 197)

# Quartiles via nearest-rank (p * n ceiling, 1-indexed) — simple, no numpy dependency.
def _quantile(xs: list[float], p: float) -> float:
    if not xs:
        raise ValueError("empty sequence")
    idx = max(0, min(len(xs) - 1, int(round(p * (len(xs) - 1)))))
    return xs[idx]

q1     = _quantile(svr_values, 0.25)
median = _quantile(svr_values, 0.50)
q3     = _quantile(svr_values, 0.75)
svr_min = svr_values[0]
svr_max = svr_values[-1]
svr_mean = sum(svr_values) / n

# Histogram bins: [0.00, 0.20), [0.20, 0.50), [0.50, 0.80), [0.80, 1.00]
BIN_EDGES = [(0.00, 0.20), (0.20, 0.50), (0.50, 0.80), (0.80, 1.01)]  # upper-inclusive on last
bin_counts = [0] * len(BIN_EDGES)
for v in svr_values:
    for i, (lo, hi) in enumerate(BIN_EDGES):
        if lo <= v < hi:
            bin_counts[i] += 1
            break
assert sum(bin_counts) == n, f"histogram lost rows: {sum(bin_counts)} vs {n}"

print()
print("=== §6.3 SVR distribution readout (observation only, n=197) ===")
print(f"  min / Q1 / median / Q3 / max : {svr_min:.4f} / {q1:.4f} / {median:.4f} / {q3:.4f} / {svr_max:.4f}")
print(f"  mean                         : {svr_mean:.4f}")
print(f"  histogram (n={n}):")
bin_labels = ["[0.00, 0.20)", "[0.20, 0.50)", "[0.50, 0.80)", "[0.80, 1.00]"]
for label, count in zip(bin_labels, bin_counts):
    pct = count / n
    bar = "#" * int(pct * 60)
    print(f"    {label:14s}  n={count:3d}  ({pct:.1%})  {bar}")


## §4 — §6.4 Fresh-7 RSI-absent vol_regime

**Claim.** Of the 7 fresh replay positions in the hard-coded fresh-7 subset `{3, 43, 68, 128, 173, 188, 198}`, **≥ 2 have SVR < 0.5**.

**Hard rule 5 binding.** The subset is the literal, pre-registered set — never re-derived from `deep_dive_candidates.json` or factor-set parsing. RSI-absence of the vol_regime is a pre-registered property of the stratum anchored in Stage 2c pos 138/143 evidence, not a per-call runtime check.

A placeholder code cell below records the RSI-absent + vol_regime-present property source (expectations.md §6.4 text + Stage 2c `call_0138_prompt.txt` / `call_0143_prompt.txt` references) as a provenance note for D8.2 audit — no runtime factor-set inspection is performed.

In [ ]:
# Cell 15 — §6.4 fresh-7 literal subset.
# Pre-registered (expectations.md §6.4, L132-138):
#   - subset: hard-coded literal {3, 43, 68, 128, 173, 188, 198}, n=7
#   - threshold: count(SVR < 0.5) over fresh-7 subset >= 2
#   - "Self-check MUST NOT re-derive this set at runtime from
#      deep_dive_candidates.json or from factor_set parsing."
# Hard rule 5 binding: subset is locked literal; this cell adjudicates
# the committed subset by position only. RSI-absence / vol_regime
# provenance is a pre-registered property of the stratum established
# in the Stage 2d selection artifacts; factor-set semantics are NOT
# reconstructed here.

FRESH_7 = (3, 43, 68, 128, 173, 188, 198)
FRESH_7_THRESHOLD = 2
check_equal("fresh-7 subset size", len(FRESH_7), 7)
check_equal("fresh-7 subset sorted uniqueness", sorted(set(FRESH_7)), list(FRESH_7))

# Verify each fresh-7 position is in the 199-call UB cohort (none quarantined).
# Expected: all 7 have critic_status == "ok" (none are 42/87/116).
fresh7_records_by_pos = {}
for pos in FRESH_7:
    matches = [r for r in per_call_records if r.get("candidate_position") == pos]
    if len(matches) != 1:
        raise RuntimeError(f"fresh-7 pos {pos}: expected 1 record, found {len(matches)}")
    fresh7_records_by_pos[pos] = matches[0]

for pos in FRESH_7:
    r = fresh7_records_by_pos[pos]
    if r.get("critic_status") != "ok":
        raise RuntimeError(f"fresh-7 pos {pos} has critic_status={r.get('critic_status')!r}, "
                           f"expected 'ok' (no fresh-7 member should be quarantined)")

print()
print("=== §6.4 fresh-7 subset membership check ===")
print(f"  locked literal subset : {FRESH_7}")
print(f"  membership check      : all 7 have critic_status=='ok' (no quarantine)")
print(f"  note                  : subset membership is locked by §6.4 and not re-derived here;")
print(f"                          the §6.4 gate adjudicates the committed subset by position.")


In [ ]:
# Cell 16 — §6.4 fresh-7 per-position table + gate adjudication.
# Threshold: count(SVR < 0.5) over fresh-7 subset >= 2 (strict <, not <=).
# Pos 188 edge case (expectations.md §6.4, L124-131): only fresh-7 member
# with factor_set_prior_occurrences >= 1; UB label agreement_expected;
# SVR expected >= 0.5 (pos 73 analog). Remains in denominator;
# falsifiability carried by the other 6 members.

# Per-position table (sorted by position ascending).
fresh7_rows = []
for pos in sorted(FRESH_7):
    r = fresh7_records_by_pos[pos]
    svr_axes = d7b_scores_by_call[str(pos)]  # all fresh-7 are "ok" → axes present
    fresh7_rows.append({
        "position"                : pos,
        "svr"                     : svr_axes["structural_variant_risk"],
        "semantic_theme_alignment": svr_axes["semantic_theme_alignment"],
        "semantic_plausibility"   : svr_axes["semantic_plausibility"],
        "pre_registered_label"    : r["pre_registered_label"],
        "candidate_theme"         : r["candidate_theme"],
        "below_threshold"         : svr_axes["structural_variant_risk"] < 0.5,
    })

fresh7_below = sum(1 for row in fresh7_rows if row["below_threshold"])
fresh7_verdict = "PASS" if fresh7_below >= FRESH_7_THRESHOLD else "FAIL"

# Every fresh-7 record should have candidate_theme == "volatility_regime"
# (structural property of the stratum; not re-derived, just recorded).
themes = {row["candidate_theme"] for row in fresh7_rows}
check_equal("fresh-7 candidate_theme set", themes, {"volatility_regime"})

print()
print("=== §6.4 fresh-7 per-position table (sorted by position) ===")
header = f"  {'pos':>4s}  {'SVR':>6s}  {'align':>6s}  {'plaus':>6s}  {'UB label':<20s}  {'<0.5':>5s}"
print(header)
print(f"  {'-'*4}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*20}  {'-'*5}")
for row in fresh7_rows:
    mark = "YES" if row["below_threshold"] else "no"
    print(f"  {row['position']:>4d}  {row['svr']:>6.4f}  "
          f"{row['semantic_theme_alignment']:>6.4f}  {row['semantic_plausibility']:>6.4f}  "
          f"{row['pre_registered_label']:<20s}  {mark:>5s}")

print()
print("=== §6.4 fresh-7 RSI-absent vol_regime gate (literal pre-registration) ===")
print(f"  pre-registered denominator : 7  (fresh-7 subset)")
print(f"  pre-registered threshold   : count(SVR < 0.5) >= {FRESH_7_THRESHOLD}")
print(f"  observed count (SVR < 0.5) : {fresh7_below}")
print(f"  observed ratio             : {fresh7_below}/7")
print(f"  verdict                    : {fresh7_verdict}")

check_equal("§6.4 fresh-7 verdict derived",
            fresh7_verdict,
            "PASS" if fresh7_below >= 2 else "FAIL")


## §5 — §6.6 Readout (observation axes only)

Per expectations.md §6.6, these axes are **recorded, not pre-registered** — no PASS/FAIL adjudication. Hard rule 4 binding.

1. **Alignment distribution** — mean, quartiles, range across 199 calls; per-axis (agreement / divergence / neutral) within UB.
2. **SVR–alignment decoupling** — counts of `svr >= 0.75 AND alignment <= 0.50` and the inverse `svr <= 0.25 AND alignment >= 0.75`. Stage 2c anchor: pos 102 (SVR 0.95 / aln 0.40) — sole decouple; no pre-registered count.
3. **Theme × label contingency** — 6 themes × 3 UB labels = 18-cell count table.
4. **Plausibility score distribution** — parallel to (1).

The neutral-stratum SVR median reported in cell 21 is observation only. Disk verification of expectations.md confirms **no numeric bound `[0.45, 0.70]` is pre-registered for Stage 2d** — the only `median` reference in expectations.md (L584) is a per-candidate MODERATE-LOW bucket test, not an aggregate neutral-median bound.

In [ ]:
# Cell 18 — §6.6(1) alignment distribution readout.
# Pre-registered (expectations.md §6.6, L149-165):
#   "Recorded, not pre-registered" — observation only, no PASS/FAIL gate.
# Axis: semantic_theme_alignment from d7b_scores_by_call.
# Denominators: 197 ok records overall, partitioned by pre_registered_label:
#   agreement=64, divergence=5, neutral=128 (64+5+128=197).

def _quantile_simple(xs: list[float], p: float) -> float:
    if not xs:
        return float("nan")
    s = sorted(xs)
    idx = max(0, min(len(s) - 1, int(round(p * (len(s) - 1)))))
    return s[idx]

def _summarize(xs: list[float]) -> dict:
    if not xs:
        return {"n": 0, "min": None, "q1": None, "median": None, "q3": None,
                "max": None, "mean": None}
    s = sorted(xs)
    return {
        "n"     : len(s),
        "min"   : s[0],
        "q1"    : _quantile_simple(s, 0.25),
        "median": _quantile_simple(s, 0.50),
        "q3"    : _quantile_simple(s, 0.75),
        "max"   : s[-1],
        "mean"  : sum(s) / len(s),
    }

alignment_overall = [row["alignment"] for row in ok_frame]
alignment_by_label = {
    "agreement_expected" : [row["alignment"] for row in ok_frame if row["pre_registered_label"] == "agreement_expected"],
    "divergence_expected": [row["alignment"] for row in ok_frame if row["pre_registered_label"] == "divergence_expected"],
    "neutral"            : [row["alignment"] for row in ok_frame if row["pre_registered_label"] == "neutral"],
}
# Sanity: partition preserves the ok_frame denominator.
check_equal("alignment partition sum",
            sum(len(v) for v in alignment_by_label.values()),
            len(alignment_overall))
check_equal("alignment partition agreement n", len(alignment_by_label["agreement_expected"]), 64)
check_equal("alignment partition divergence n", len(alignment_by_label["divergence_expected"]), 5)
check_equal("alignment partition neutral n", len(alignment_by_label["neutral"]), 128)

def _print_summary(label: str, xs: list[float]) -> None:
    s = _summarize(xs)
    if s["n"] == 0:
        print(f"  {label:<22s}  n=0")
        return
    print(f"  {label:<22s}  n={s['n']:>3d}  min={s['min']:.4f}  "
          f"Q1={s['q1']:.4f}  median={s['median']:.4f}  Q3={s['q3']:.4f}  "
          f"max={s['max']:.4f}  mean={s['mean']:.4f}")

print()
print("=== §6.6(1) alignment distribution — OBSERVATION ONLY, NOT A GATE ===")
_print_summary("overall (n=197)", alignment_overall)
_print_summary("agreement (n=64)", alignment_by_label["agreement_expected"])
_print_summary("divergence (n=5)", alignment_by_label["divergence_expected"])
_print_summary("neutral (n=128)",  alignment_by_label["neutral"])
print("  note                  : §6.6 is recorded, not pre-registered; no PASS/FAIL adjudication.")


In [ ]:
# Cell 19 — §6.6(2) SVR-alignment decoupling readout.
# Pre-registered (expectations.md §6.6, L155-158):
#   "count of calls where svr >= 0.75 AND alignment <= 0.50 and inverse"
#   "Stage 2c anchor: pos 102 (SVR 0.95 / aln 0.40) is the sole Stage 2c
#    decouple; no pre-registered count."
# Observation only — no pre-registered bound.
# Denominator: n=197 ok_frame.

# Decouple A: high SVR + low alignment (the canonical Stage 2c pos-102-style decouple).
decouple_high_svr_low_aln = [
    row for row in ok_frame if row["svr"] >= 0.75 and row["alignment"] <= 0.50
]
# Decouple B: inverse — low SVR + high alignment.
decouple_low_svr_high_aln = [
    row for row in ok_frame if row["svr"] <= 0.25 and row["alignment"] >= 0.75
]

check_equal("SVR-alignment decouple denominator", len(ok_frame), 197)

print()
print("=== §6.6(2) SVR-alignment decoupling — OBSERVATION ONLY, NOT A GATE ===")
print(f"  denominator                     : 197 (ok scoring universe)")
print(f"  (A) SVR >= 0.75 AND aln <= 0.50 : n={len(decouple_high_svr_low_aln)}")
for row in sorted(decouple_high_svr_low_aln, key=lambda r: r["candidate_position"]):
    print(f"      pos {row['candidate_position']:3d}  svr={row['svr']:.4f}  aln={row['alignment']:.4f}  "
          f"label={row['pre_registered_label']}  theme={row['candidate_theme']}")
print(f"  (B) SVR <= 0.25 AND aln >= 0.75 : n={len(decouple_low_svr_high_aln)}")
for row in sorted(decouple_low_svr_high_aln, key=lambda r: r["candidate_position"]):
    print(f"      pos {row['candidate_position']:3d}  svr={row['svr']:.4f}  aln={row['alignment']:.4f}  "
          f"label={row['pre_registered_label']}  theme={row['candidate_theme']}")
print("  note                            : Stage 2c anchor pos 102 (SVR 0.95 / aln 0.40) was the sole Stage 2c")
print("                                    decouple of type (A); no pre-registered count — observation only.")


In [ ]:
# Cell 20 — §6.6(3) theme × label contingency readout.
# Pre-registered (expectations.md §6.6, L159-160):
#   "Theme × label contingency — 6 themes × 3 labels = 18-cell count table;
#    read-out only."
# Canonical Stage 2d theme list (agents/proposer/stage2d_batch.py L8) uses
# 5 themes, not 6; the "6 themes" phrasing in §6.6 is nominal framing.
# Observation only — no PASS/FAIL adjudication.
# Denominator: 199 UB cohort (pos 116 excluded per §6.1 Lock; pos 42 and 87
# remain in the cohort and carry valid theme + label metadata).

# Canonical Stage 2d themes (5) and UB labels (3).
STAGE2D_THEMES = ("momentum", "mean_reversion", "volatility_regime",
                  "volume_divergence", "calendar_effect")
UB_LABELS = ("agreement_expected", "divergence_expected", "neutral")

# UB cohort: 199 records (pos 116 excluded).
ub_cohort_for_contingency = [
    r for r in per_call_records
    if not (r.get("critic_status") == "skipped_source_invalid"
            and r.get("position") == 116)
]
check_equal("§6.6 contingency UB denominator", len(ub_cohort_for_contingency), 199)

# Build 5×3 contingency table.
contingency = {(t, l): 0 for t in STAGE2D_THEMES for l in UB_LABELS}
for r in ub_cohort_for_contingency:
    t = r.get("candidate_theme")
    l = r.get("pre_registered_label")
    if (t, l) in contingency:
        contingency[(t, l)] += 1
    else:
        # Unknown theme/label combo — flag loudly (should not happen).
        raise RuntimeError(f"unexpected (theme={t!r}, label={l!r}) in UB cohort")

# Sanity: populated cells sum to 199.
total = sum(contingency.values())
check_equal("§6.6 contingency cell-count sum", total, 199)

# Row/column marginals.
row_totals = {t: sum(contingency[(t, l)] for l in UB_LABELS) for t in STAGE2D_THEMES}
col_totals = {l: sum(contingency[(t, l)] for t in STAGE2D_THEMES) for l in UB_LABELS}
check_equal("§6.6 contingency agreement col", col_totals["agreement_expected"], 66)
check_equal("§6.6 contingency divergence col", col_totals["divergence_expected"], 5)
check_equal("§6.6 contingency neutral col", col_totals["neutral"], 128)

print()
print("=== §6.6(3) theme × label contingency — OBSERVATION ONLY, NOT A GATE ===")
print(f"  denominator : 199 (UB cohort, pos 116 excluded)")
print(f"  shape       : 5 themes × 3 labels = 15 cells (Stage 2d canonical; §6.6 'six themes'")
print(f"                is nominal framing, live data has 5 themes).")
print()
header = f"  {'theme':<20s}  {'agreement':>10s}  {'divergence':>11s}  {'neutral':>8s}  {'row total':>10s}"
print(header)
print(f"  {'-'*20}  {'-'*10}  {'-'*11}  {'-'*8}  {'-'*10}")
for t in STAGE2D_THEMES:
    a = contingency[(t, "agreement_expected")]
    d = contingency[(t, "divergence_expected")]
    n = contingency[(t, "neutral")]
    print(f"  {t:<20s}  {a:>10d}  {d:>11d}  {n:>8d}  {row_totals[t]:>10d}")
print(f"  {'-'*20}  {'-'*10}  {'-'*11}  {'-'*8}  {'-'*10}")
print(f"  {'col total':<20s}  {col_totals['agreement_expected']:>10d}  "
      f"{col_totals['divergence_expected']:>11d}  {col_totals['neutral']:>8d}  {total:>10d}")


In [ ]:
# Cell 21 — §6.6 neutral-stratum SVR median + IQR readout.
# OBSERVATION ONLY — NOT a pre-registered gate.
# The neutral-stratum "[0.45, 0.70]" range sometimes referenced in Stage 2c
# descriptive text is NOT pre-registered for Stage 2d per expectations.md.
# Hard rule 5: do not re-interpret non-pre-registered claims as gates.
# Denominator: neutral cohort in ok_frame (n=128).

neutral_svrs = sorted(row["svr"] for row in ok_frame if row["pre_registered_label"] == "neutral")
check_equal("neutral SVR readout denominator", len(neutral_svrs), 128)

def _q(xs: list[float], p: float) -> float:
    if not xs:
        return float("nan")
    idx = max(0, min(len(xs) - 1, int(round(p * (len(xs) - 1)))))
    return xs[idx]

neutral_min    = neutral_svrs[0]
neutral_q1     = _q(neutral_svrs, 0.25)
neutral_median = _q(neutral_svrs, 0.50)
neutral_q3     = _q(neutral_svrs, 0.75)
neutral_max    = neutral_svrs[-1]
neutral_mean   = sum(neutral_svrs) / len(neutral_svrs)
neutral_iqr    = neutral_q3 - neutral_q1

print()
print("=== §6.6 neutral-stratum SVR readout — OBSERVATION ONLY, NOT A GATE ===")
print(f"  denominator : 128 (ok_frame, pre_registered_label == 'neutral')")
print(f"  min         : {neutral_min:.4f}")
print(f"  Q1          : {neutral_q1:.4f}")
print(f"  median      : {neutral_median:.4f}")
print(f"  Q3          : {neutral_q3:.4f}")
print(f"  max         : {neutral_max:.4f}")
print(f"  IQR (Q3-Q1) : {neutral_iqr:.4f}")
print(f"  mean        : {neutral_mean:.4f}")
print()
print("  note        : this readout is observation only. The neutral-stratum [0.45, 0.70]")
print("                range referenced in Stage 2c descriptive text is NOT a Stage 2d")
print("                pre-registered bound; no PASS/FAIL verdict is derived here.")
print("                Any interpretive framing (comparison to Stage 2c, anchor analysis,")
print("                etc.) is D8.2 scope, not D8.1.")


## §6 — Stratum / theme cross-tabs (forensic observation)

Forensic readout of theme × UB label × SVR bucket. Per Lock 3.6 and §6.5 ruling, no theme-stratified threshold is pre-registered; this section is recorded for D8.2 interpretive input and carries no pass/fail.

Per Q3 adjudication (Round 2 ratification): report direct counts and within-row percentages only. **No binomial p-value. No Wilson CI. No hypothesis-test framing.** Columns: theme, UB label, SVR bucket, count, row denominator, within-row percentage.

In [ ]:
# Cell 23 — theme × UB label × SVR bucket cross-tab.
# Direct counts + within-row percentage ONLY. No CI. No p-value.
# Placeholder only.

# SVR_BUCKETS = ["LOW (<0.25)", "MOD-LOW [0.25,0.50)", "MOD-HIGH [0.50,0.85)", "HIGH [0.85,1.0]"]
# rows: theme, ub_label, svr_bucket, count, row_denominator, within_row_pct


## §7 — Synthesis

Assemble the per-claim gate adjudication table, the research implication matrix, and the final D8.1 verdict. Downstream D8.2 consumes this section's summary artifacts.

In [ ]:
# Cell 25 — per-claim PASS/FAIL summary table.
# Placeholder only.

# summary = [
#     {"claim": "§6.2.1 agreement",       "observed": ..., "threshold": "≥52/66 at SVR≥0.5", "verdict": ...},
#     {"claim": "§6.2.2 divergence",      "observed": ..., "threshold": "≥4/5 at SVR≥0.5",  "verdict": ...},
#     {"claim": "§6.3(a) upper tail",     "observed": ..., "threshold": "≥90/199 at SVR≥0.80", "verdict": ...},
#     {"claim": "§6.3(b) lower tail",     "observed": ..., "threshold": "≥30/199 at SVR≤0.30", "verdict": ...},
#     {"claim": "§6.4 fresh-7",           "observed": ..., "threshold": "≥2/7 at SVR<0.5",  "verdict": ...},
# ]
# render as table


## §7.1 — Research implication matrix

Seven rows, claim-per-row. Each row names a finding, the observed result, its pre-registered status (gate vs readout), and the D8 follow-up implication. No scenario-combinatoric rows.

| # | finding | observed_result | pre_registered_status | d8_followup |
|---|---|---|---|---|
| 1 | §6.2.1 agreement axis | _filled after cell 09 runs_ | gate | carry verdict into D8.2 axis narrative |
| 2 | §6.2.2 divergence axis | _filled after cell 10 runs_ | gate | if FAIL, D8.2 owns axis-reversal interpretation question |
| 3 | §6.3(a) upper-tail distribution | _filled after cell 12 runs_ | gate | carry into D8.2 distributional-shape narrative |
| 4 | §6.3(b) lower-tail distribution | _filled after cell 12 runs_ | gate | carry into D8.2 distributional-shape narrative |
| 5 | §6.4 fresh-7 RSI-absent vol_regime | _filled after cell 16 runs_ | gate | feeds D8.3 test-retest context for pos 138/143 |
| 6 | §6.6 observation axes (alignment / decouple / theme × label / plausibility / neutral median) | _filled after cells 18–21 run_ | readout | recorded as forensic input to D8.2; no gate adjudication |
| 7 | D8.2 / D8.3 carry-forward | aggregated from rows 1–6 | n/a | §E3 Tier 1/2 bucket adjudication → D8.2; test-retest + drift → D8.3 |

## §7.2 — Final D8.1 analytical verdict

Final D8.1 analytical verdict will be computed after Sections 2–7 execute.

Gate claims are §6.2.1, §6.2.2, §6.3(a), §6.3(b), and §6.4.

§6.6 is readout-only.